In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense, Reshape, Flatten, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation, ZeroPadding2D,Conv2DTranspose

In [2]:
EPOCHS=50
BATCH_SIZE=128
NOISE_DIM=100
num_examples_to_generate=16

In [3]:
(x_train, _), (_, _) = mnist.load_data()

In [4]:
x_train=x_train.reshape(x_train.shape[0],28,28,1).astype('float32')
x_train=(x_train-127.5)/127.5

In [5]:
dataset=tf.data.Dataset.from_tensor_slices(x_train)
dataset=dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE)

In [7]:

def make_generator_model():
    model=Sequential()
    model.add(Dense(7*7*256,use_bias=False,input_shape=(NOISE_DIM,)))
    model.add(BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((7,7,256)))
    assert model.output_shape==(None,7,7,256)

    model.add(Conv2DTranspose(128,(5,5),strides=(1,1),padding="same",use_bias=False))
    assert model.output_shape==(None,7,7,128)
    model.add(BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(Conv2DTranspose(64,(5,5),strides=(2,2),padding="same",use_bias=False))
    assert model.output_shape==(None,14,14,64)
    model.add(BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(Conv2DTranspose(1,(5,5),strides=(2,2),padding="same",use_bias=False,activation="tanh"))
    assert model.output_shape==(None,28,28,1)
    model.add(BatchNormalization())
    model.add(layers.LeakyReLU())
    return model
